# VolGAN: A Generative Model for Arbitrage-Free Implied Volatility Surfaces

**Paper**: Vuletić & Cont (2024), *Applied Mathematical Finance*, 31(4), 203–238.

---

## 1. Paper Summary

### What is the problem?
Options are quoted via **implied volatility (IV)** surfaces — a 2D function σ(m, τ) of moneyness *m = K/S* and time-to-maturity *τ*. Any realistic model of IV dynamics must simultaneously:

1. **Satisfy static arbitrage constraints** — call prices must be increasing in τ, decreasing in m, and convex in m.
2. **Capture empirical co-movement structure** — a small number of PCA factors (level, skew, curvature), negative correlation with the underlying ("leverage effect"), non-Gaussian increments, time-varying correlations.
3. **Be useful for hedging and risk management** — generate scenarios that lead to good hedge ratios.

Parametric models struggle to satisfy all three requirements simultaneously given the high dimensionality of the surface.

### What does VolGAN do?
VolGAN is a **Conditional GAN** that learns to generate one-day-ahead joint scenarios for:
- The **log-return** of the underlying index
- The **log-increment** of the entire IV surface on a (moneyness × maturity) grid

given a conditioning vector containing recent returns, realized volatility, and the current IV surface.

### Key innovations
| Component | Description |
|---|---|
| **Custom loss** | BCE + discrete Sobolev smoothness penalties (L_m, L_τ) on the generated surface |
| **Gradient norm matching** | Automatic calibration of regularization weights α_m, α_τ |
| **Scenario re-weighting** | Exponential tilting via arbitrage penalty to suppress arbitrage-violating scenarios |
| **Regression hedging** | Hedge ratios via OLS/LASSO regression across VolGAN scenarios |

### Key results (SPX data, 2000–2023)
- Generates smooth, near-arbitrage-free surfaces (penalty ≈ market data levels)
- Learns PCA structure: first 3 PCs match data (level, skew, curvature)
- Produces non-Gaussian, fat-tailed return distributions
- Learns time-varying correlations between returns and IV
- VolGAN + LASSO hedging **outperforms** BS delta and delta-vega hedging (lower VaR, tighter tracking error)

### Pros & Cons

| Pros | Cons |
|---|---|
| Fully data-driven, model-free | Requires large historical IV dataset (OptionMetrics) |
| Handles non-Gaussian dynamics & time-varying correlations | One-step-ahead only; multi-step requires sequential generation |
| Arbitrage constraints via soft penalty + re-weighting | Re-weighting can collapse effective sample size in turbulent periods |
| Flexible hedging instrument selection via LASSO | Small network (16–32 neurons) may limit expressiveness |
| Stable over 4.5 years out-of-sample | Slight overestimation of OTM call IV |
| Open-source code on GitHub | BCE loss (vs. Wasserstein) — mode collapse risk |


## 2. Implementation

### 2.1 Imports and Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import cm
from mpl_toolkits.mplot3d import Axes3D
import torch
import torch.nn as nn
import torch.optim as optim
from scipy.stats import norm
import warnings
warnings.filterwarnings('ignore')

# Reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")


### 2.2 Black-Scholes Pricing and IV Surface Utilities

The implied volatility σ(m, τ) satisfies:

$$C(m, \tau) = C_{BS}(S, K, \tau, \sigma) = S \, N(d_1) - K e^{-r\tau} N(d_2)$$

where $d_1 = \frac{-\ln m + \tau(r + \sigma^2/2)}{\sigma\sqrt{\tau}}$ and $d_2 = d_1 - \sigma\sqrt{\tau}$.


In [ ]:
def bs_call_price(S, K, tau, sigma, r=0.0):
    """Black-Scholes call price."""
    if tau <= 0:
        return max(S - K, 0.0)
    d1 = (-np.log(K / S) + tau * (r + sigma**2 / 2)) / (sigma * np.sqrt(tau))
    d2 = d1 - sigma * np.sqrt(tau)
    return S * norm.cdf(d1) - K * np.exp(-r * tau) * norm.cdf(d2)


def bs_call_price_vec(S, K, tau, sigma, r=0.0):
    """Vectorized Black-Scholes call price."""
    sigma = np.maximum(sigma, 1e-8)
    tau = np.maximum(tau, 1e-8)
    d1 = (-np.log(K / S) + tau * (r + sigma**2 / 2)) / (sigma * np.sqrt(tau))
    d2 = d1 - sigma * np.sqrt(tau)
    return S * norm.cdf(d1) - K * np.exp(-r * tau) * norm.cdf(d2)


def relative_call_price(m, tau, sigma, r=0.0):
    """Relative call price c(m, tau) = C_BS / S (Eq. 1 in paper)."""
    sigma = np.maximum(sigma, 1e-8)
    tau = np.maximum(tau, 1e-8)
    d1 = (-np.log(m) + tau * (r + sigma**2 / 2)) / (sigma * np.sqrt(tau))
    d2 = d1 - sigma * np.sqrt(tau)
    return norm.cdf(d1) - m * np.exp(-r * tau) * norm.cdf(d2)


# Moneyness and maturity grids (from the paper)
MONEYNESS_GRID = np.array([0.6, 0.7, 0.8, 0.9, 0.95, 1.0, 1.05, 1.1, 1.2, 1.3, 1.4])
MATURITY_GRID = np.array([1/252, 1/52, 2/52, 1/12, 1/6, 1/4, 1/2, 3/4, 1.0])

Nm = len(MONEYNESS_GRID)
Ntau = len(MATURITY_GRID)
print(f"Grid size: {Nm} moneyness x {Ntau} maturities = {Nm * Ntau} surface points")


### 2.3 Arbitrage Penalty (Eq. 2–5)

Static arbitrage constraints require call prices to be:
- **Calendar spread**: increasing in τ → p₁
- **Call spread**: decreasing in m → p₂  
- **Butterfly spread**: convex in m → p₃

The total arbitrage penalty is Ψ(σ) = p₁ + p₂ + p₃.


In [ ]:
def arbitrage_penalty(sigma_surface, m_grid=MONEYNESS_GRID, tau_grid=MATURITY_GRID, r=0.0):
    """
    Compute arbitrage penalty Ψ(σ) = p1 + p2 + p3 (Eq. 2-5).
    sigma_surface: shape (Nm, Ntau)
    """
    Nm, Ntau = sigma_surface.shape
    
    # Compute relative call prices on the grid
    c = np.zeros((Nm, Ntau))
    for i in range(Nm):
        for j in range(Ntau):
            c[i, j] = relative_call_price(m_grid[i], tau_grid[j], sigma_surface[i, j], r)
    
    # p1: Calendar arbitrage (Eq. 3) — call prices should increase in tau
    p1 = 0.0
    for i in range(Nm):
        for j in range(Ntau - 1):
            dtau = tau_grid[j+1] - tau_grid[j]
            violation = tau_grid[j] * (c[i, j] - c[i, j+1]) / dtau
            p1 += max(0.0, violation)
    
    # p2: Call spread (Eq. 4) — call prices should decrease in moneyness
    p2 = 0.0
    for i in range(Nm - 1):
        for j in range(Ntau):
            dm = m_grid[i+1] - m_grid[i]
            violation = (c[i+1, j] - c[i, j]) / dm
            p2 += max(0.0, violation)
    
    # p3: Butterfly arbitrage (Eq. 5) — call prices should be convex in moneyness
    p3 = 0.0
    for i in range(1, Nm - 1):
        for j in range(Ntau):
            dm_left = m_grid[i] - m_grid[i-1]
            dm_right = m_grid[i+1] - m_grid[i]
            left_slope = (c[i, j] - c[i-1, j]) / dm_left
            right_slope = (c[i+1, j] - c[i, j]) / dm_right
            violation = left_slope - right_slope
            p3 += max(0.0, violation)
    
    return p1 + p2 + p3, p1, p2, p3


# Quick test with a flat IV surface
test_surface = np.full((Nm, Ntau), 0.20)
psi, p1, p2, p3 = arbitrage_penalty(test_surface)
print(f"Flat 20% IV surface -> Ψ = {psi:.6f} (p1={p1:.6f}, p2={p2:.6f}, p3={p3:.6f})")
print("A flat surface should have near-zero penalty (small numerical effects only).")


### 2.4 Data: Synthetic IV Surface Generation

Since OptionMetrics data requires a commercial license, we generate **synthetic but realistic** IV surface time series that mimic key empirical properties of SPX options:

- Volatility smile/skew (higher IV for low strikes)
- Term structure (typically upward sloping)
- Level dynamics correlated with underlying returns (leverage effect)
- Mean-reverting volatility
- Non-Gaussian increments

We use a simplified stochastic model to generate training data, then train VolGAN on it.


In [ ]:
def generate_synthetic_iv_surface(m_grid, tau_grid, base_vol=0.20, skew=-0.15, 
                                   smile=0.05, term_slope=0.02):
    """Generate a single realistic IV surface with smile, skew, and term structure."""
    Nm, Ntau = len(m_grid), len(tau_grid)
    sigma = np.zeros((Nm, Ntau))
    for i, m in enumerate(m_grid):
        for j, tau in enumerate(tau_grid):
            log_m = np.log(m)
            # Skew + smile + term structure
            sigma[i, j] = (base_vol 
                          + skew * log_m                    # skew
                          + smile * log_m**2                # smile/convexity
                          + term_slope * np.sqrt(tau)       # term structure
                          )
    sigma = np.clip(sigma, 0.03, 1.5)  # Physical bounds
    return sigma


def generate_iv_time_series(n_days=3000, m_grid=MONEYNESS_GRID, tau_grid=MATURITY_GRID,
                            seed=42):
    """
    Generate synthetic daily time series of:
    - S_t: underlying price
    - sigma_t(m, tau): implied volatility surface
    
    Mimics key SPX empirical properties.
    """
    np.random.seed(seed)
    Nm, Ntau = len(m_grid), len(tau_grid)
    
    # Parameters
    S0 = 4000.0
    mu = 0.05 / 252          # daily drift
    base_vol0 = 0.18
    kappa = 0.03             # mean reversion speed for base vol
    theta = 0.18             # long-run mean vol
    xi = 0.015               # vol-of-vol
    rho = -0.75              # leverage effect correlation
    skew0 = -0.15
    smile0 = 0.05
    term_slope0 = 0.02
    
    dt = 1.0 / 252
    
    # Storage
    S = np.zeros(n_days)
    log_returns = np.zeros(n_days)
    surfaces = np.zeros((n_days, Nm, Ntau))
    base_vols = np.zeros(n_days)
    
    S[0] = S0
    base_vols[0] = base_vol0
    surfaces[0] = generate_synthetic_iv_surface(m_grid, tau_grid, base_vol0, skew0, smile0, term_slope0)
    
    for t in range(1, n_days):
        # Correlated Brownian motions
        z1 = np.random.randn()
        z2 = rho * z1 + np.sqrt(1 - rho**2) * np.random.randn()
        
        # Surface noise (correlated across grid, driven by ~3 factors)
        n_factors = 3
        factor_loadings = np.random.randn(n_factors)
        factor_vols = np.array([0.008, 0.005, 0.003])
        
        # Mean-reverting base vol (Heston-like)
        base_vols[t] = base_vols[t-1] + kappa * (theta - base_vols[t-1]) * dt + xi * np.sqrt(dt) * z2
        base_vols[t] = max(0.05, min(0.80, base_vols[t]))
        
        # Dynamic skew — becomes more negative when vol is high
        dynamic_skew = skew0 - 0.1 * (base_vols[t] - theta)
        
        # Underlying return
        sigma_atm = base_vols[t]
        log_returns[t] = mu * dt + sigma_atm * np.sqrt(dt) * z1
        S[t] = S[t-1] * np.exp(log_returns[t])
        
        # Generate surface with perturbations
        surfaces[t] = generate_synthetic_iv_surface(
            m_grid, tau_grid, 
            base_vols[t], dynamic_skew, 
            smile0 + 0.01 * np.random.randn(),
            term_slope0 + 0.005 * np.random.randn()
        )
        
        # Add small correlated noise across surface
        noise = np.zeros((Nm, Ntau))
        for k in range(n_factors):
            pattern = np.outer(
                np.sin(np.linspace(0, (k+1)*np.pi, Nm)),
                np.cos(np.linspace(0, (k+0.5)*np.pi, Ntau))
            )
            noise += factor_vols[k] * factor_loadings[k] * pattern
        
        surfaces[t] += noise * np.sqrt(dt)
        surfaces[t] = np.clip(surfaces[t], 0.03, 1.5)
    
    return S, log_returns, surfaces, base_vols


# Generate data
print("Generating synthetic IV surface time series...")
S_prices, log_rets, iv_surfaces, base_vols = generate_iv_time_series(n_days=3000)
print(f"Generated {len(S_prices)} days of data")
print(f"Surface shape per day: {iv_surfaces[0].shape}")
print(f"Total surface points per day: {iv_surfaces[0].size}")


### 2.5 Visualize Synthetic Data

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Price path
axes[0, 0].plot(S_prices, linewidth=0.8)
axes[0, 0].set_title('Synthetic Underlying Price')
axes[0, 0].set_xlabel('Day')
axes[0, 0].set_ylabel('Price')

# Plot 2: Log returns
axes[0, 1].plot(log_rets[1:] * np.sqrt(252), linewidth=0.5, alpha=0.7)
axes[0, 1].set_title('Annualized Daily Log Returns')
axes[0, 1].set_xlabel('Day')
axes[0, 1].set_ylabel('Return')

# Plot 3: ATM vol time series
atm_idx = np.argmin(np.abs(MONEYNESS_GRID - 1.0))
tau_3m_idx = np.argmin(np.abs(MATURITY_GRID - 0.25))
axes[1, 0].plot(iv_surfaces[:, atm_idx, tau_3m_idx], linewidth=0.8)
axes[1, 0].set_title('3-month ATM Implied Volatility')
axes[1, 0].set_xlabel('Day')
axes[1, 0].set_ylabel('IV')

# Plot 4: Sample IV surface (3D)
ax3d = fig.add_subplot(2, 2, 4, projection='3d')
M, T = np.meshgrid(MONEYNESS_GRID, MATURITY_GRID, indexing='ij')
ax3d.plot_surface(M, T, iv_surfaces[1500], cmap=cm.viridis, alpha=0.8)
ax3d.set_xlabel('Moneyness')
ax3d.set_ylabel('Time to Maturity')
ax3d.set_zlabel('IV')
ax3d.set_title('Sample IV Surface (Day 1500)')

plt.tight_layout()
plt.savefig('synthetic_data_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print("Data overview plotted.")


### 2.6 Feature Engineering (Condition Vector)

Following the paper (Eq. 6–9), for each day *t* we construct:

**State variables (log-space):**
$$g_t(m, \tau) = \log \sigma_t(m, \tau), \quad \Delta g_t = g_{t+\Delta t} - g_t$$

**Condition vector:**
$$a_t = (R_{t-\Delta t},\; R_{t-2\Delta t},\; \gamma_{t-\Delta t},\; g_t(m, \tau))$$

where $\gamma_t$ is the 1-month realized volatility and $R_t = \log(S_{t+\Delta t}/S_t)$.

**Target:** $(R_t, \Delta g_t(m, \tau))$


In [ ]:
def compute_realized_vol(log_returns, window=21):
    """1-month realized volatility (Eq. 8)."""
    rv = np.zeros(len(log_returns))
    for t in range(window, len(log_returns)):
        rv[t] = np.sqrt(252 / window * np.sum(log_returns[t-window:t]**2))
    return rv


def prepare_volgan_dataset(S, log_returns, iv_surfaces, m_grid, tau_grid):
    """
    Prepare condition vectors and targets for VolGAN training.
    
    Returns:
        conditions: list of (R_{t-1}, R_{t-2}, gamma_{t-1}, g_t) vectors
        targets: list of (R_t, Delta_g_t) vectors
    """
    n_days = len(S)
    Nm, Ntau = len(m_grid), len(tau_grid)
    surface_dim = Nm * Ntau
    
    # Log IV surfaces
    log_iv = np.log(np.clip(iv_surfaces, 1e-6, None))
    
    # Log IV increments
    delta_log_iv = np.diff(log_iv, axis=0)  # shape: (n_days-1, Nm, Ntau)
    
    # Realized volatility
    rv = compute_realized_vol(log_returns, window=21)
    
    conditions = []
    targets = []
    
    # Start from day 22 (need 21 days for realized vol + 2 lagged returns)
    for t in range(22, n_days - 1):
        # Condition: (R_{t-1}, R_{t-2}, gamma_{t-1}, g_t)
        cond = np.concatenate([
            [log_returns[t-1]],          # R_{t-1}
            [log_returns[t-2]],          # R_{t-2}
            [rv[t-1]],                   # gamma_{t-1}
            log_iv[t].flatten()          # g_t(m, tau) flattened
        ])
        
        # Target: (R_t, Delta_g_t)
        tgt = np.concatenate([
            [log_returns[t]],            # R_t
            delta_log_iv[t].flatten()    # Delta g_t flattened
        ])
        
        conditions.append(cond)
        targets.append(tgt)
    
    conditions = np.array(conditions, dtype=np.float32)
    targets = np.array(targets, dtype=np.float32)
    
    return conditions, targets


# Prepare dataset
conditions, targets = prepare_volgan_dataset(S_prices, log_rets, iv_surfaces, 
                                              MONEYNESS_GRID, MATURITY_GRID)

cond_dim = conditions.shape[1]
target_dim = targets.shape[1]
surface_dim = Nm * Ntau

print(f"Dataset size: {len(conditions)} samples")
print(f"Condition dim: {cond_dim} (2 returns + 1 RV + {surface_dim} surface points)")
print(f"Target dim: {target_dim} (1 return + {surface_dim} surface increments)")

# Train/test split (80/20)
split_idx = int(0.8 * len(conditions))
train_cond, test_cond = conditions[:split_idx], conditions[split_idx:]
train_tgt, test_tgt = targets[:split_idx], targets[split_idx:]
print(f"Train: {len(train_cond)}, Test: {len(test_cond)}")


### 2.7 VolGAN Architecture (PyTorch)

**Generator** (Figure 1 in paper):
- Input: condition vector $a_t$ + noise $z \sim N(0, I_d)$, $d=32$
- 3-layer feedforward: H → 2H → output_dim
- Activations: softplus, softplus, linear
- Output: $(\hat{R}_t(z), \Delta\hat{g}_t(m, \tau)(z))$

**Discriminator** (Figure 2 in paper):
- Input: condition $a_t$ + sample $(R, \Delta g)$ (real or fake)
- 2-layer feedforward: H → 1
- Activations: softplus, sigmoid
- Output: probability that input is real data


In [ ]:
class VolGANGenerator(nn.Module):
    """
    Generator G(a_t, z_t) -> (R_hat_t, Delta_g_hat_t)
    
    Architecture from Section 3.4:
    - noise_dim (d) = 32
    - H = 16
    - 3 layers: H, 2H, output_dim
    - Activations: softplus, softplus, linear
    """
    def __init__(self, cond_dim, noise_dim=32, hidden_dim=16, output_dim=100):
        super().__init__()
        self.noise_dim = noise_dim
        
        input_dim = cond_dim + noise_dim
        
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.Softplus(),
            nn.Linear(hidden_dim, 2 * hidden_dim),
            nn.Softplus(),
            nn.Linear(2 * hidden_dim, output_dim),  # affine (no activation)
        )
    
    def forward(self, condition, noise):
        """
        Args:
            condition: (batch, cond_dim) — the a_t vector
            noise: (batch, noise_dim) — z_t ~ N(0, I)
        Returns:
            output: (batch, output_dim) — (R_hat, Delta_g_hat)
        """
        x = torch.cat([condition, noise], dim=1)
        return self.net(x)


class VolGANDiscriminator(nn.Module):
    """
    Discriminator D(a_t, (R, Delta_g)) -> [0, 1]
    
    Architecture from Section 3.4:
    - H = 16
    - 2 layers: H, 1
    - Activations: softplus, sigmoid
    """
    def __init__(self, cond_dim, sample_dim):
        super().__init__()
        
        input_dim = cond_dim + sample_dim
        hidden_dim = 16
        
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.Softplus(),
            nn.Linear(hidden_dim, 1),
            nn.Sigmoid(),
        )
    
    def forward(self, condition, sample):
        """
        Args:
            condition: (batch, cond_dim)
            sample: (batch, sample_dim) — either real (R, Δg) or generator output
        Returns:
            prob: (batch, 1) — probability that sample is real
        """
        x = torch.cat([condition, sample], dim=1)
        return self.net(x)


# Instantiate
noise_dim = 32
G = VolGANGenerator(cond_dim, noise_dim=noise_dim, hidden_dim=16, output_dim=target_dim).to(device)
D = VolGANDiscriminator(cond_dim, target_dim).to(device)

print("Generator:")
print(f"  Parameters: {sum(p.numel() for p in G.parameters()):,}")
print(f"  Input: condition({cond_dim}) + noise({noise_dim}) = {cond_dim + noise_dim}")
print(f"  Output: {target_dim} (1 return + {surface_dim} IV increments)")
print()
print("Discriminator:")
print(f"  Parameters: {sum(p.numel() for p in D.parameters()):,}")
print(f"  Input: condition({cond_dim}) + sample({target_dim}) = {cond_dim + target_dim}")
print(f"  Output: 1 (real/fake probability)")


### 2.8 Smoothness Penalties (Eq. 11–12)

The key innovation: discrete Sobolev semi-norms that penalize roughness in both moneyness and maturity directions:

$$L_m(g) = \sum_{i,j} \frac{(g(m_{i+1}, \tau_j) - g(m_i, \tau_j))^2}{|m_{i+1} - m_i|^2}$$

$$L_\tau(g) = \sum_{i,j} \frac{(g(m_i, \tau_{j+1}) - g(m_i, \tau_j))^2}{|\tau_{j+1} - \tau_j|^2}$$

These are applied to the **simulated log-IV surface** $\hat{g}_t = g_t + \Delta\hat{g}_t$.


In [ ]:
def smoothness_penalty_m(log_iv_surface, m_grid):
    """
    L_m: smoothness penalty in moneyness direction (Eq. 11).
    log_iv_surface: (batch, Nm, Ntau) tensor
    """
    dm = torch.tensor(np.diff(m_grid), dtype=torch.float32, device=log_iv_surface.device)
    # Differences along moneyness dimension
    diff_m = log_iv_surface[:, 1:, :] - log_iv_surface[:, :-1, :]  # (batch, Nm-1, Ntau)
    # Normalize by grid spacing squared
    penalty = (diff_m / dm.unsqueeze(0).unsqueeze(-1))**2
    return penalty.sum(dim=(1, 2)).mean()


def smoothness_penalty_tau(log_iv_surface, tau_grid):
    """
    L_tau: smoothness penalty in maturity direction (Eq. 12).
    log_iv_surface: (batch, Nm, Ntau) tensor
    """
    dtau = torch.tensor(np.diff(tau_grid), dtype=torch.float32, device=log_iv_surface.device)
    # Differences along maturity dimension
    diff_tau = log_iv_surface[:, :, 1:] - log_iv_surface[:, :, :-1]  # (batch, Nm, Ntau-1)
    # Normalize by grid spacing squared
    penalty = (diff_tau / dtau.unsqueeze(0).unsqueeze(0))**2
    return penalty.sum(dim=(1, 2)).mean()


# Test on a sample surface
test_surf = torch.tensor(np.log(iv_surfaces[100:102]), dtype=torch.float32, device=device)
print(f"L_m = {smoothness_penalty_m(test_surf, MONEYNESS_GRID):.4f}")
print(f"L_tau = {smoothness_penalty_tau(test_surf, MATURITY_GRID):.4f}")


### 2.9 Training: Gradient Norm Matching + Full Training

**Phase 1** (Section 3.4): Train with BCE only for `n_grad=25` epochs. Record gradient norms of BCE, L_m, L_τ w.r.t. θ_g. Set α_m, α_τ as mean ratios.

**Phase 2**: Train with full loss (Eq. 13) for `n_epochs` epochs.

$$J^{(G)} = -\frac{1}{2}\mathbb{E}[\log D(a_t, G(a_t, z_t))] + \alpha_m \mathbb{E}[L_m(\hat{g}_t)] + \alpha_\tau \mathbb{E}[L_\tau(\hat{g}_t)]$$

$$J^{(D)} = -\frac{1}{2}\mathbb{E}[\log D(a_t, \text{real})] - \frac{1}{2}\mathbb{E}[\log(1 - D(a_t, G(a_t, z_t)))]$$


In [ ]:
def extract_surface_from_target(target_batch, current_log_iv, Nm, Ntau):
    """
    Given generator output (R_hat, Delta_g_hat), reconstruct the simulated log-IV surface.
    
    target_batch: (batch, 1 + Nm*Ntau)
    current_log_iv: (batch, Nm*Ntau) — g_t from the condition vector
    
    Returns: (batch, Nm, Ntau) — simulated log-IV surface g_hat_t
    """
    delta_g = target_batch[:, 1:]  # skip the return component
    g_hat = current_log_iv + delta_g
    return g_hat.view(-1, Nm, Ntau)


def train_volgan(G, D, train_cond, train_tgt, 
                 n_grad_epochs=25, n_full_epochs=500,
                 batch_size=100, noise_dim=32, lr=1e-4,
                 Nm=Nm, Ntau=Ntau, m_grid=MONEYNESS_GRID, tau_grid=MATURITY_GRID):
    """
    Full VolGAN training procedure:
    1. Gradient norm matching phase
    2. Full training with smoothness penalties
    """
    
    opt_G = optim.RMSprop(G.parameters(), lr=lr)
    opt_D = optim.RMSprop(D.parameters(), lr=lr)
    
    n_samples = len(train_cond)
    surface_start_idx = 3  # condition = [R_{t-1}, R_{t-2}, gamma, g_t...]
    
    # Convert to tensors
    cond_tensor = torch.tensor(train_cond, dtype=torch.float32, device=device)
    tgt_tensor = torch.tensor(train_tgt, dtype=torch.float32, device=device)
    
    eps = 1e-8  # for numerical stability in log
    
    # =============================================
    # PHASE 1: Gradient Norm Matching
    # =============================================
    print("Phase 1: Gradient Norm Matching...")
    bce_norms = []
    lm_norms = []
    ltau_norms = []
    
    for epoch in range(n_grad_epochs):
        perm = torch.randperm(n_samples)
        epoch_bce_norms = []
        epoch_lm_norms = []
        epoch_ltau_norms = []
        
        for start in range(0, n_samples - batch_size, batch_size):
            idx = perm[start:start + batch_size]
            cond_batch = cond_tensor[idx]
            real_batch = tgt_tensor[idx]
            
            # Current log-IV from condition
            current_log_iv = cond_batch[:, surface_start_idx:]
            
            # === Train Discriminator ===
            noise = torch.randn(batch_size, noise_dim, device=device)
            fake_batch = G(cond_batch, noise)
            
            d_real = D(cond_batch, real_batch)
            d_fake = D(cond_batch, fake_batch.detach())
            
            loss_D = -0.5 * torch.mean(torch.log(d_real + eps)) - 0.5 * torch.mean(torch.log(1 - d_fake + eps))
            
            opt_D.zero_grad()
            loss_D.backward()
            opt_D.step()
            
            # === Compute Generator gradient norms (BCE only) ===
            noise = torch.randn(batch_size, noise_dim, device=device)
            fake_batch = G(cond_batch, noise)
            d_fake = D(cond_batch, fake_batch)
            
            # BCE term
            loss_bce = -0.5 * torch.mean(torch.log(d_fake + eps))
            opt_G.zero_grad()
            loss_bce.backward(retain_graph=True)
            bce_grad_norm = sum(p.grad.norm()**2 for p in G.parameters() if p.grad is not None).sqrt().item()
            
            # L_m term
            sim_surface = extract_surface_from_target(fake_batch, current_log_iv, Nm, Ntau)
            loss_lm = smoothness_penalty_m(sim_surface, m_grid)
            opt_G.zero_grad()
            loss_lm.backward(retain_graph=True)
            lm_grad_norm = sum(p.grad.norm()**2 for p in G.parameters() if p.grad is not None).sqrt().item()
            
            # L_tau term
            sim_surface = extract_surface_from_target(fake_batch, current_log_iv, Nm, Ntau)
            loss_ltau = smoothness_penalty_tau(sim_surface, tau_grid)
            opt_G.zero_grad()
            loss_ltau.backward()
            ltau_grad_norm = sum(p.grad.norm()**2 for p in G.parameters() if p.grad is not None).sqrt().item()
            
            epoch_bce_norms.append(bce_grad_norm)
            epoch_lm_norms.append(max(lm_grad_norm, eps))
            epoch_ltau_norms.append(max(ltau_grad_norm, eps))
            
            # Update generator with BCE only in phase 1
            noise = torch.randn(batch_size, noise_dim, device=device)
            fake_batch = G(cond_batch, noise)
            d_fake = D(cond_batch, fake_batch)
            loss_G = -0.5 * torch.mean(torch.log(d_fake + eps))
            opt_G.zero_grad()
            loss_G.backward()
            opt_G.step()
        
        bce_norms.extend(epoch_bce_norms)
        lm_norms.extend(epoch_lm_norms)
        ltau_norms.extend(epoch_ltau_norms)
    
    # Compute alpha_m, alpha_tau as mean ratios
    ratios_m = [b / l for b, l in zip(bce_norms, lm_norms)]
    ratios_tau = [b / l for b, l in zip(bce_norms, ltau_norms)]
    alpha_m = np.mean(ratios_m)
    alpha_tau = np.mean(ratios_tau)
    print(f"  alpha_m = {alpha_m:.4f}, alpha_tau = {alpha_tau:.4f}")
    
    # =============================================
    # PHASE 2: Full Training
    # =============================================
    print(f"\nPhase 2: Full Training ({n_full_epochs} epochs)...")
    
    # Re-initialize (from same init — in practice would save/reload)
    G_state = G.state_dict()
    D_state = D.state_dict()
    
    history = {'G_loss': [], 'D_loss': [], 'L_m': [], 'L_tau': []}
    
    for epoch in range(n_full_epochs):
        perm = torch.randperm(n_samples)
        epoch_G, epoch_D, epoch_Lm, epoch_Lt = [], [], [], []
        
        for start in range(0, n_samples - batch_size, batch_size):
            idx = perm[start:start + batch_size]
            cond_batch = cond_tensor[idx]
            real_batch = tgt_tensor[idx]
            current_log_iv = cond_batch[:, surface_start_idx:]
            
            # === Train Discriminator (Eq. 14) ===
            noise = torch.randn(batch_size, noise_dim, device=device)
            fake_batch = G(cond_batch, noise).detach()
            
            d_real = D(cond_batch, real_batch)
            d_fake = D(cond_batch, fake_batch)
            
            loss_D = -0.5 * torch.mean(torch.log(d_real + eps)) - 0.5 * torch.mean(torch.log(1 - d_fake + eps))
            
            opt_D.zero_grad()
            loss_D.backward()
            opt_D.step()
            
            # === Train Generator (Eq. 13) ===
            noise = torch.randn(batch_size, noise_dim, device=device)
            fake_batch = G(cond_batch, noise)
            d_fake = D(cond_batch, fake_batch)
            
            # BCE term
            loss_bce = -0.5 * torch.mean(torch.log(d_fake + eps))
            
            # Smoothness penalties on simulated surfaces
            sim_surface = extract_surface_from_target(fake_batch, current_log_iv, Nm, Ntau)
            loss_lm = smoothness_penalty_m(sim_surface, m_grid)
            loss_ltau = smoothness_penalty_tau(sim_surface, tau_grid)
            
            # Full generator loss
            loss_G = loss_bce + alpha_m * loss_lm + alpha_tau * loss_ltau
            
            opt_G.zero_grad()
            loss_G.backward()
            opt_G.step()
            
            epoch_G.append(loss_G.item())
            epoch_D.append(loss_D.item())
            epoch_Lm.append(loss_lm.item())
            epoch_Lt.append(loss_ltau.item())
        
        history['G_loss'].append(np.mean(epoch_G))
        history['D_loss'].append(np.mean(epoch_D))
        history['L_m'].append(np.mean(epoch_Lm))
        history['L_tau'].append(np.mean(epoch_Lt))
        
        if (epoch + 1) % 100 == 0 or epoch == 0:
            print(f"  Epoch {epoch+1:4d} | G_loss: {history['G_loss'][-1]:.4f} | "
                  f"D_loss: {history['D_loss'][-1]:.4f} | "
                  f"L_m: {history['L_m'][-1]:.4f} | L_tau: {history['L_tau'][-1]:.4f}")
    
    return history, alpha_m, alpha_tau


# Train VolGAN
history, alpha_m, alpha_tau = train_volgan(
    G, D, train_cond, train_tgt,
    n_grad_epochs=25, n_full_epochs=500,
    batch_size=100, noise_dim=noise_dim, lr=1e-4
)
print("\nTraining complete!")


### 2.10 Training Curves

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

axes[0, 0].plot(history['G_loss'], linewidth=0.8)
axes[0, 0].set_title('Generator Loss')
axes[0, 0].set_xlabel('Epoch')

axes[0, 1].plot(history['D_loss'], linewidth=0.8, color='orange')
axes[0, 1].set_title('Discriminator Loss')
axes[0, 1].set_xlabel('Epoch')

axes[1, 0].plot(history['L_m'], linewidth=0.8, color='green')
axes[1, 0].set_title('Smoothness Penalty L_m')
axes[1, 0].set_xlabel('Epoch')

axes[1, 1].plot(history['L_tau'], linewidth=0.8, color='red')
axes[1, 1].set_title('Smoothness Penalty L_τ')
axes[1, 1].set_xlabel('Epoch')

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()


### 2.11 Scenario Generation & Re-Weighting (Section 3.3)

Given a condition $a_t$, generate $N=10{,}000$ samples from the generator, then apply exponential tilting:

$$w_i = \frac{\exp(-\beta \, \Psi(\hat{\sigma}_i))}{\sum_j \exp(-\beta \, \Psi(\hat{\sigma}_j))}$$

with adaptive $\beta(t) = 500 / \max_i \{w_i(t)\}$ (Eq. 20).


In [ ]:
@torch.no_grad()
def generate_scenarios(G, condition, n_samples=1000, noise_dim=32):
    """Generate n_samples scenarios from the trained generator."""
    G.eval()
    cond = torch.tensor(condition, dtype=torch.float32, device=device).unsqueeze(0)
    cond = cond.expand(n_samples, -1)
    noise = torch.randn(n_samples, noise_dim, device=device)
    output = G(cond, noise)
    G.train()
    return output.cpu().numpy()


def scenario_reweighting(scenarios, current_log_iv, m_grid, tau_grid, beta=None):
    """
    Apply scenario re-weighting (Eq. 15-17).
    
    scenarios: (N, 1 + Nm*Ntau) — generated (R, Delta_g) samples
    current_log_iv: (Nm*Ntau,) — current log-IV surface (flattened)
    
    Returns: weights (N,), scenarios (N, ...)
    """
    N = len(scenarios)
    Nm, Ntau = len(m_grid), len(tau_grid)
    
    penalties = np.zeros(N)
    for i in range(N):
        delta_g = scenarios[i, 1:]
        log_iv_sim = current_log_iv + delta_g
        iv_sim = np.exp(log_iv_sim).reshape(Nm, Ntau)
        pen, _, _, _ = arbitrage_penalty(iv_sim, m_grid, tau_grid)
        penalties[i] = pen
    
    if beta is None:
        # Adaptive beta (Eq. 20): first compute uniform weights, then scale
        uniform_weights = np.exp(-penalties)
        uniform_weights /= uniform_weights.sum()
        beta = 500.0 / max(uniform_weights.max(), 1e-10)
    
    log_weights = -beta * penalties
    log_weights -= log_weights.max()  # numerical stability
    weights = np.exp(log_weights)
    weights /= weights.sum()
    
    return weights, penalties


# Generate scenarios for a test condition
test_idx = 0
test_condition = test_cond[test_idx]
current_log_iv = test_condition[3:]  # g_t from condition

print("Generating 1000 scenarios...")
scenarios = generate_scenarios(G, test_condition, n_samples=1000, noise_dim=noise_dim)

print("Applying scenario re-weighting...")
weights, penalties = scenario_reweighting(scenarios, current_log_iv, MONEYNESS_GRID, MATURITY_GRID)

print(f"\nScenario statistics:")
print(f"  Mean arbitrage penalty (raw): {np.mean(penalties):.6f}")
print(f"  Weighted mean arb. penalty:   {np.sum(weights * penalties):.6f}")
print(f"  Effective sample size:        {1.0 / np.sum(weights**2):.1f} / {len(weights)}")
print(f"  Max weight:                   {weights.max():.6f}")


### 2.12 Visualize Generated vs Real Surfaces

In [ ]:
fig = plt.figure(figsize=(18, 5))
M, T = np.meshgrid(MONEYNESS_GRID, MATURITY_GRID, indexing='ij')

# Real next-day surface
real_delta_g = test_tgt[test_idx, 1:]
real_log_iv_next = current_log_iv + real_delta_g
real_iv_next = np.exp(real_log_iv_next).reshape(Nm, Ntau)

ax1 = fig.add_subplot(131, projection='3d')
ax1.plot_surface(M, T, real_iv_next, cmap=cm.viridis, alpha=0.8)
ax1.set_title('Real Next-Day Surface')
ax1.set_xlabel('Moneyness')
ax1.set_ylabel('TTM')
ax1.set_zlabel('IV')

# Sample VolGAN output (unweighted mean)
mean_delta_g = np.mean(scenarios[:, 1:], axis=0)
mean_log_iv = current_log_iv + mean_delta_g
mean_iv = np.exp(mean_log_iv).reshape(Nm, Ntau)

ax2 = fig.add_subplot(132, projection='3d')
ax2.plot_surface(M, T, mean_iv, cmap=cm.viridis, alpha=0.8)
ax2.set_title('VolGAN Mean Surface')
ax2.set_xlabel('Moneyness')
ax2.set_ylabel('TTM')
ax2.set_zlabel('IV')

# Weighted mean
wmean_delta_g = np.average(scenarios[:, 1:], axis=0, weights=weights)
wmean_log_iv = current_log_iv + wmean_delta_g
wmean_iv = np.exp(wmean_log_iv).reshape(Nm, Ntau)

ax3 = fig.add_subplot(133, projection='3d')
ax3.plot_surface(M, T, wmean_iv, cmap=cm.viridis, alpha=0.8)
ax3.set_title('VolGAN Weighted Mean')
ax3.set_xlabel('Moneyness')
ax3.set_ylabel('TTM')
ax3.set_zlabel('IV')

plt.tight_layout()
plt.savefig('generated_vs_real_surfaces.png', dpi=150, bbox_inches='tight')
plt.show()


### 2.13 Out-of-Sample Forecasting (Section 4.2.3)

Generate next-day forecasts with 95% confidence intervals using the 2.5% and 97.5% quantiles of the scenario distribution.


In [ ]:
def oos_forecast(G, test_cond, test_tgt, n_scenarios=500, noise_dim=32):
    """Generate one-step-ahead forecasts on test set."""
    n_test = len(test_cond)
    atm_idx = np.argmin(np.abs(MONEYNESS_GRID - 1.0))
    tau_3m_idx = np.argmin(np.abs(MATURITY_GRID - 0.25))
    surface_offset = atm_idx * Ntau + tau_3m_idx
    
    forecasts = {
        'return_mean': [], 'return_q025': [], 'return_q975': [],
        'atm3m_mean': [], 'atm3m_q025': [], 'atm3m_q975': [],
        'return_real': [], 'atm3m_real': []
    }
    
    for t in range(0, n_test, 5):  # every 5th day for speed
        scenarios = generate_scenarios(G, test_cond[t], n_scenarios, noise_dim)
        
        # Returns
        sim_returns = scenarios[:, 0]
        forecasts['return_mean'].append(np.mean(sim_returns))
        forecasts['return_q025'].append(np.percentile(sim_returns, 2.5))
        forecasts['return_q975'].append(np.percentile(sim_returns, 97.5))
        forecasts['return_real'].append(test_tgt[t, 0])
        
        # 3-month ATM vol
        current_log_iv = test_cond[t, 3:]
        sim_log_iv = current_log_iv + scenarios[:, 1:]
        sim_atm3m = np.exp(sim_log_iv[:, surface_offset])
        
        real_atm3m = np.exp(current_log_iv[surface_offset] + test_tgt[t, 1 + surface_offset])
        
        forecasts['atm3m_mean'].append(np.mean(sim_atm3m))
        forecasts['atm3m_q025'].append(np.percentile(sim_atm3m, 2.5))
        forecasts['atm3m_q975'].append(np.percentile(sim_atm3m, 97.5))
        forecasts['atm3m_real'].append(real_atm3m)
    
    return {k: np.array(v) for k, v in forecasts.items()}


print("Running out-of-sample forecasts...")
fc = oos_forecast(G, test_cond, test_tgt, n_scenarios=500)
print(f"Forecast points: {len(fc['return_real'])}")

# Plot
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

t_axis = np.arange(len(fc['return_real']))

# Returns
axes[0].fill_between(t_axis, fc['return_q025'], fc['return_q975'], alpha=0.3, color='blue', label='95% CI')
axes[0].plot(t_axis, fc['return_real'], '.', markersize=2, color='red', label='Realized')
axes[0].plot(t_axis, fc['return_mean'], linewidth=0.5, color='blue', alpha=0.5, label='Forecast mean')
axes[0].set_title('Next-Day Return Forecast')
axes[0].legend()
axes[0].set_xlabel('Test day (sampled)')

# ATM vol
axes[1].fill_between(t_axis, fc['atm3m_q025'], fc['atm3m_q975'], alpha=0.3, color='blue', label='95% CI')
axes[1].plot(t_axis, fc['atm3m_real'], '.', markersize=2, color='red', label='Realized')
axes[1].plot(t_axis, fc['atm3m_mean'], linewidth=0.5, color='blue', alpha=0.5, label='Forecast mean')
axes[1].set_title('3-Month ATM IV Forecast')
axes[1].legend()
axes[1].set_xlabel('Test day (sampled)')

plt.tight_layout()
plt.savefig('oos_forecasts.png', dpi=150, bbox_inches='tight')
plt.show()

# Coverage
return_coverage = np.mean((fc['return_real'] >= fc['return_q025']) & 
                           (fc['return_real'] <= fc['return_q975']))
atm_coverage = np.mean((fc['atm3m_real'] >= fc['atm3m_q025']) & 
                         (fc['atm3m_real'] <= fc['atm3m_q975']))
print(f"\n95% CI coverage — Returns: {return_coverage:.1%}, 3M ATM Vol: {atm_coverage:.1%}")


### 2.14 Principal Component Analysis (Section 4.2.5)

The paper shows VolGAN learns the covariance structure: PCA on simulated IV increments should yield factors matching real data (level, skew, curvature).


In [ ]:
from numpy.linalg import eigh

def pca_on_increments(data_matrix, n_components=3):
    """PCA on rows of data_matrix. Returns eigenvalues, eigenvectors, variance explained."""
    cov = np.cov(data_matrix, rowvar=False)
    eigenvalues, eigenvectors = eigh(cov)
    # Sort descending
    idx = np.argsort(eigenvalues)[::-1]
    eigenvalues = eigenvalues[idx]
    eigenvectors = eigenvectors[:, idx]
    total_var = eigenvalues.sum()
    var_explained = eigenvalues[:n_components] / total_var * 100
    return eigenvalues[:n_components], eigenvectors[:, :n_components], var_explained


# Real test data increments
real_increments = test_tgt[:, 1:]  # Delta g (log-IV increments), shape (n_test, Nm*Ntau)

# Simulated increments: generate one scenario per test condition
print("Generating simulated increments for PCA...")
sim_increments = []
for t in range(len(test_cond)):
    sc = generate_scenarios(G, test_cond[t], n_samples=1, noise_dim=noise_dim)
    sim_increments.append(sc[0, 1:])
sim_increments = np.array(sim_increments)

# PCA
evals_real, evecs_real, var_real = pca_on_increments(real_increments)
evals_sim, evecs_sim, var_sim = pca_on_increments(sim_increments)

print(f"\nVariance explained by top 3 PCs:")
print(f"  Real data:  PC1={var_real[0]:.1f}%, PC2={var_real[1]:.1f}%, PC3={var_real[2]:.1f}%")
print(f"  VolGAN:     PC1={var_sim[0]:.1f}%, PC2={var_sim[1]:.1f}%, PC3={var_sim[2]:.1f}%")

# Inner products (Table 5 in paper)
for k in range(3):
    ip = abs(np.dot(evecs_real[:, k], evecs_sim[:, k]))
    print(f"  |<PC{k+1}_data, PC{k+1}_sim>| = {ip:.4f}")

# Visualize PCs as surfaces
fig, axes = plt.subplots(2, 3, figsize=(16, 8), subplot_kw={'projection': '3d'})

for k in range(3):
    pc_real = evecs_real[:, k].reshape(Nm, Ntau)
    pc_sim = evecs_sim[:, k].reshape(Nm, Ntau)
    
    axes[0, k].plot_surface(M, T, pc_real, cmap=cm.RdBu, alpha=0.8)
    axes[0, k].set_title(f'Data PC{k+1} ({var_real[k]:.1f}%)')
    axes[0, k].set_xlabel('m')
    axes[0, k].set_ylabel('τ')
    
    axes[1, k].plot_surface(M, T, pc_sim, cmap=cm.RdBu, alpha=0.8)
    axes[1, k].set_title(f'VolGAN PC{k+1} ({var_sim[k]:.1f}%)')
    axes[1, k].set_xlabel('m')
    axes[1, k].set_ylabel('τ')

plt.suptitle('Principal Components: Data (top) vs VolGAN (bottom)', fontsize=14)
plt.tight_layout()
plt.savefig('pca_comparison.png', dpi=150, bbox_inches='tight')
plt.show()


### 2.15 Correlation Structure (Section 4.2.6)

The paper demonstrates VolGAN captures:
- Negative correlation between returns and IV level changes (leverage effect)
- Time-varying instantaneous correlations


In [ ]:
# Correlations between simulated returns and IV increments
sim_returns = []
sim_atm_changes = []

atm_idx = np.argmin(np.abs(MONEYNESS_GRID - 1.0))
tau_1m_idx = np.argmin(np.abs(MATURITY_GRID - 1/12))
surface_offset_1m = atm_idx * Ntau + tau_1m_idx

for t in range(min(500, len(test_cond))):
    sc = generate_scenarios(G, test_cond[t], n_samples=100, noise_dim=noise_dim)
    # Per-scenario correlation
    sim_returns.append(sc[:, 0].mean())
    sim_atm_changes.append(sc[:, 1 + surface_offset_1m].mean())

sim_returns = np.array(sim_returns)
sim_atm_changes = np.array(sim_atm_changes)

# Real data
real_returns = test_tgt[:500, 0]
real_atm_changes = test_tgt[:500, 1 + surface_offset_1m]

corr_real = np.corrcoef(real_returns, real_atm_changes)[0, 1]
corr_sim = np.corrcoef(sim_returns, sim_atm_changes)[0, 1]

print(f"Correlation (return vs 1M ATM IV change):")
print(f"  Real data: {corr_real:.4f}")
print(f"  VolGAN:    {corr_sim:.4f}")
print(f"  Both should be negative (leverage effect)")

# Rolling correlation to show time-varying nature
window = 50
rolling_corr_real = pd.Series(real_returns).rolling(window).corr(pd.Series(real_atm_changes))
rolling_corr_sim = pd.Series(sim_returns).rolling(window).corr(pd.Series(sim_atm_changes))

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(rolling_corr_real.values, label='Real data', alpha=0.8)
ax.plot(rolling_corr_sim.values, label='VolGAN simulated', alpha=0.8)
ax.axhline(y=0, color='k', linestyle='--', alpha=0.3)
ax.set_title('Rolling 50-day Correlation: Return vs 1M ATM IV Change')
ax.set_xlabel('Test day')
ax.set_ylabel('Correlation')
ax.legend()
plt.tight_layout()
plt.savefig('correlation_structure.png', dpi=150, bbox_inches='tight')
plt.show()


### 2.16 Application: Scenario-Based Regression Hedging (Section 5)

The paper's most practical application: use VolGAN scenarios to compute hedge ratios by regressing simulated portfolio PnL on hedging instrument PnL across scenarios (Eq. 27).

We demonstrate with a simplified hedging example:
- **Target**: ATM straddle (call + put, m=1, τ=1/12)
- **Hedging instruments**: underlying (delta hedge) + VolGAN regression
- **Comparison**: BS delta hedge vs VolGAN scenario-based regression


In [ ]:
from numpy.linalg import lstsq

def bs_delta(S, K, tau, sigma, r=0.0):
    """Black-Scholes call delta."""
    if tau <= 1e-8:
        return 1.0 if S > K else 0.0
    d1 = (-np.log(K / S) + tau * (r + sigma**2 / 2)) / (sigma * np.sqrt(tau))
    return norm.cdf(d1)


def hedging_backtest(G, test_cond, test_tgt, iv_surfaces_test, S_test,
                     n_scenarios=500, noise_dim=32):
    """
    Simplified hedging backtest comparing BS delta vs VolGAN regression.
    
    Target: ATM call option with ~1 month maturity.
    Hedge: underlying only, rebalanced daily.
    """
    n_test = min(len(test_cond), 200)  # limit for speed
    
    atm_idx = np.argmin(np.abs(MONEYNESS_GRID - 1.0))
    tau_idx = np.argmin(np.abs(MATURITY_GRID - 1/12))
    
    bs_errors = []
    volgan_errors = []
    
    for t in range(n_test - 1):
        # Current state
        S_t = S_test[t]
        K = S_t  # ATM
        tau_t = MATURITY_GRID[tau_idx]
        sigma_t = iv_surfaces_test[t, atm_idx, tau_idx]
        
        # Option value today
        V_t = bs_call_price(S_t, K, tau_t, sigma_t)
        
        # Next day
        S_t1 = S_test[t + 1]
        sigma_t1 = iv_surfaces_test[t + 1, atm_idx, tau_idx]
        tau_t1 = max(tau_t - 1/252, 1e-6)
        V_t1 = bs_call_price(S_t1, K, tau_t1, sigma_t1)
        
        dV = V_t1 - V_t
        dS = S_t1 - S_t
        
        # BS delta hedge
        delta_bs = bs_delta(S_t, K, tau_t, sigma_t)
        bs_pnl = dV - delta_bs * dS
        bs_errors.append(bs_pnl)
        
        # VolGAN regression hedge
        scenarios = generate_scenarios(G, test_cond[t], n_scenarios, noise_dim)
        
        # Simulate S changes
        sim_dS = S_t * (np.exp(scenarios[:, 0]) - 1)
        
        # Simulate option value changes
        sim_delta_log_iv = scenarios[:, 1 + atm_idx * Ntau + tau_idx]
        current_log_iv_val = test_cond[t, 3 + atm_idx * Ntau + tau_idx]
        sim_sigma = np.exp(current_log_iv_val + sim_delta_log_iv)
        sim_V = np.array([bs_call_price(S_t * np.exp(scenarios[i, 0]), K, tau_t1, sim_sigma[i]) 
                          for i in range(n_scenarios)])
        sim_dV = sim_V - V_t
        
        # Regression: dV = alpha + phi * dS + epsilon
        A = np.column_stack([np.ones(n_scenarios), sim_dS])
        result = lstsq(A, sim_dV, rcond=None)
        phi_volgan = result[0][1]  # regression hedge ratio
        
        volgan_pnl = dV - phi_volgan * dS
        volgan_errors.append(volgan_pnl)
    
    return np.array(bs_errors), np.array(volgan_errors)


# Reconstruct test surfaces and prices
test_start = split_idx + 22  # offset for the dataset preparation
S_test = S_prices[test_start:test_start + len(test_cond)]
iv_test = iv_surfaces[test_start:test_start + len(test_cond)]

print("Running hedging backtest...")
bs_err, vg_err = hedging_backtest(G, test_cond, test_tgt, iv_test, S_test,
                                   n_scenarios=300, noise_dim=noise_dim)

print(f"\nHedging Error Statistics:")
print(f"{'':20s} {'BS Delta':>12s} {'VolGAN Reg':>12s}")
print(f"{'Mean':20s} {np.mean(bs_err):12.4f} {np.mean(vg_err):12.4f}")
print(f"{'Std Dev':20s} {np.std(bs_err):12.4f} {np.std(vg_err):12.4f}")
print(f"{'5% VaR':20s} {-np.percentile(bs_err, 5):12.4f} {-np.percentile(vg_err, 5):12.4f}")
print(f"{'1% VaR':20s} {-np.percentile(bs_err, 1):12.4f} {-np.percentile(vg_err, 1):12.4f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(bs_err, linewidth=0.5, alpha=0.7, label='BS Delta')
axes[0].plot(vg_err, linewidth=0.5, alpha=0.7, label='VolGAN')
axes[0].legend()
axes[0].set_title('Tracking Error Over Time')
axes[0].set_xlabel('Day')
axes[0].set_ylabel('PnL ($)')

axes[1].hist(bs_err, bins=50, alpha=0.5, density=True, label='BS Delta')
axes[1].hist(vg_err, bins=50, alpha=0.5, density=True, label='VolGAN')
axes[1].legend()
axes[1].set_title('Tracking Error Distribution')
axes[1].set_xlabel('PnL ($)')
axes[1].set_ylabel('Density')

plt.tight_layout()
plt.savefig('hedging_comparison.png', dpi=150, bbox_inches='tight')
plt.show()


### 2.17 Non-Gaussian Distribution Analysis (Section 4.2.4)

The paper shows VolGAN generates non-Gaussian, asymmetric distributions with fat tails — a property difficult to capture with parametric models.


In [ ]:
# Generate many scenarios from a single condition to analyze distributions
test_day = 100
scenarios_large = generate_scenarios(G, test_cond[test_day], n_samples=5000, noise_dim=noise_dim)

sim_rets = scenarios_large[:, 0] * np.sqrt(252)  # annualized
real_rets = test_tgt[:, 0] * np.sqrt(252)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Return distribution
axes[0].hist(sim_rets, bins=80, density=True, alpha=0.6, label='VolGAN simulated', color='steelblue')
x = np.linspace(sim_rets.min(), sim_rets.max(), 200)
axes[0].plot(x, norm.pdf(x, sim_rets.mean(), sim_rets.std()), 'r--', 
             linewidth=2, label='Gaussian fit')
axes[0].set_title('Distribution of Simulated Returns')
axes[0].set_xlabel('Annualized Return')
axes[0].set_ylabel('Density')
axes[0].legend()
axes[0].set_yscale('log')

# ATM vol increment distribution
atm_idx = np.argmin(np.abs(MONEYNESS_GRID - 1.0))
tau_1m_idx = np.argmin(np.abs(MATURITY_GRID - 1/12))
offset = atm_idx * Ntau + tau_1m_idx

sim_vol_incr = scenarios_large[:, 1 + offset]
axes[1].hist(sim_vol_incr, bins=80, density=True, alpha=0.6, label='VolGAN simulated', color='steelblue')
x2 = np.linspace(sim_vol_incr.min(), sim_vol_incr.max(), 200)
axes[1].plot(x2, norm.pdf(x2, sim_vol_incr.mean(), sim_vol_incr.std()), 'r--',
             linewidth=2, label='Gaussian fit')
axes[1].set_title('Distribution of 1M ATM Vol Increments')
axes[1].set_xlabel('Δ log σ')
axes[1].set_ylabel('Density')
axes[1].legend()
axes[1].set_yscale('log')

plt.tight_layout()
plt.savefig('distribution_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

# Skewness and kurtosis
from scipy.stats import skew, kurtosis
print(f"Return distribution: skew={skew(sim_rets):.3f}, excess kurtosis={kurtosis(sim_rets):.3f}")
print(f"Vol increment dist:  skew={skew(sim_vol_incr):.3f}, excess kurtosis={kurtosis(sim_vol_incr):.3f}")
print("(Gaussian: skew=0, excess kurtosis=0)")


## 3. Summary & Conclusions

### What we implemented
1. **Black-Scholes pricing utilities** and IV surface parameterization
2. **Arbitrage penalty** (calendar, call spread, butterfly — Eq. 2–5)
3. **VolGAN Generator & Discriminator** in PyTorch (Section 3.1, 3.4)
4. **Smoothness penalties** L_m, L_τ as discrete Sobolev semi-norms (Eq. 11–12)
5. **Two-phase training** with gradient norm matching for α_m, α_τ calibration
6. **Scenario generation & re-weighting** with adaptive β (Section 3.3)
7. **Out-of-sample forecasting** with confidence intervals
8. **PCA analysis** of generated IV increments
9. **Correlation structure** analysis (leverage effect)
10. **Scenario-based regression hedging** vs BS delta (Section 5)
11. **Distribution analysis** showing non-Gaussian tails

### Key takeaways from the paper

| Aspect | Finding |
|---|---|
| Surface quality | Smoothness penalties are essential; BCE-only GAN produces irregular surfaces |
| Arbitrage | Re-weighting reduces arb violations to market data levels |
| PCA structure | First 3 PCs match data: level (50%), skew (25–34%), curvature (5–13%) |
| Correlations | Time-varying, strongly negative return-vol correlation learned |
| Distributions | Non-Gaussian, fat-tailed, asymmetric — cannot be captured by Brownian models |
| Hedging | VolGAN + LASSO: lowest VaR, mean tracking error ≈ 0 |
| Robustness | Stable 4.5 years OOS, including Covid-19 and Ukraine war periods |

### Limitations & future directions
- **Multi-step simulation**: current model is one-step; iterative application accumulates errors
- **Data requirements**: needs dense, clean IV surface data (OptionMetrics quality)
- **BCE loss**: susceptible to mode collapse; Wasserstein-GAN or spectral normalization could help
- **Surface grid**: fixed grid; could extend to continuous parameterization (e.g., neural SDE)
- **Transaction costs**: hedging analysis doesn't include costs; LASSO selection partially addresses via sparsity

### References
- Vuletić & Cont (2024). VolGAN. *Applied Mathematical Finance*, 31(4), 203–238.
- Code: [github.com/milenavuletic/VolGAN](https://github.com/milenavuletic/VolGAN/)
